# Homework 3

## 1 - Model selection for (polynomial) regression

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

seed = 67

data = pd.read_csv("poly_dataset.csv", sep=",", skiprows=1)
X = data.iloc[:, :-1]
y = data.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

In [26]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold

model = Pipeline([
    ("polynomial", PolynomialFeatures()),
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

# The double underscores are Pipeline syntax
param_grid = {
    "polynomial__degree": [1, 2, 3, 4, 5, 6],
    "ridge__alpha": [0, 0.001, 0.01, 0.1]
}

kf = KFold(n_splits=5, shuffle=True, random_state=seed)

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="neg_mean_squared_error", # Negative because look for the maximum score
    refit=True,
    cv=kf
)

grid_search.fit(X_train, y_train)

for i in range(len(param_grid["polynomial__degree"]) * len(param_grid["ridge__alpha"])):
    d = grid_search.cv_results_["param_polynomial__degree"][i]
    a = grid_search.cv_results_["param_ridge__alpha"][i]
    mean = grid_search.cv_results_["mean_test_score"][i]
    print(f"d={d}, a={a}, mean={mean}")

print(grid_search.best_params_)
print(f"5-fold CV gives (mean) MSE: {-max(grid_search.cv_results_["mean_test_score"])} on validation data.")

d=1, a=0.0, mean=-148.8088811301636
d=1, a=0.001, mean=-148.8088747165643
d=1, a=0.01, mean=-148.80881707154055
d=1, a=0.1, mean=-148.80824827855537
d=2, a=0.0, mean=-1.1981671042754483
d=2, a=0.001, mean=-1.1981668893894222
d=2, a=0.01, mean=-1.1981653480026002
d=2, a=0.1, mean=-1.1981887220171614
d=3, a=0.0, mean=-1.049400263226765
d=3, a=0.001, mean=-1.0493873816554413
d=3, a=0.01, mean=-1.0492763128589038
d=3, a=0.1, mean=-1.0486265243549338
d=4, a=0.0, mean=-1.8541053639780163
d=4, a=0.001, mean=-1.0819438087545739
d=4, a=0.01, mean=-1.081256563514921
d=4, a=0.1, mean=-1.079919916683269
d=5, a=0.0, mean=-1.3031871757434381
d=5, a=0.001, mean=-1.123200384258845
d=5, a=0.01, mean=-1.1161039832001642
d=5, a=0.1, mean=-1.1084937001037498
d=6, a=0.0, mean=-1.6827438562724695
d=6, a=0.001, mean=-1.1520668225001116
d=6, a=0.01, mean=-1.1338864544740679
d=6, a=0.1, mean=-1.1164875407382095
{'polynomial__degree': 3, 'ridge__alpha': 0.1}
5-fold CV gives (mean) MSE: 1.0486265243549338 on val

In [20]:
# Testing my best model on the test set
from sklearn.metrics import mean_squared_error

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
MSE = mean_squared_error(y_test, y_pred)

print(f"5-fold CV gives MSE: {MSE} on the test data.")

5-fold CV gives MSE: 0.9167483347821226 on the test data.


In [21]:
X_train_resplit, X_val, y_train_resplit, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=seed)

dimensions = [1, 2, 3, 4, 5, 6]
alphas = [0, 0.001, 0.01, 0.1]

best_score = np.inf
best_params = None

for d in dimensions:
    for a in alphas:
        model = Pipeline([
            ("polynomial", PolynomialFeatures(d)),
            ("scaler", StandardScaler()),
            ("ridge", Ridge(a)),
        ])

        model.fit(X_train_resplit, y_train_resplit)
        y_pred = model.predict(X_val)
        MSE = mean_squared_error(y_val, y_pred)

        if MSE < best_score:
            best_score = MSE
            best_params = (d, a)

print(f"dim={best_params[0]}, alpha={best_params[1]}")
print(f"Manual splitting gives MSE: {best_score} on validation data.")

dim=3, alpha=0.1
Manual splitting gives MSE: 1.2049875699111383 on validation data.


In [22]:
d, a = best_params

model = Pipeline([
    ("polynomial", PolynomialFeatures(d)),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(a)),
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
MSE = mean_squared_error(y_test, y_pred)

print(f"Manual splitting gives MSE: {MSE} on test data.")

Manual splitting gives MSE: 0.9167483347821226 on test data.


On question 1.6  
**How does MSE compare for the two methods?**  
MSE for the two methods is identical as they both end up picking the exact same hyperparameters.
For all seeds I tested it picked dimensions=3 and alpha=0.1 every time.

On question 1.7  
**Why is standardizing your features important when doing ridge regression?**  
Ridge regression's goal is to penalize large coefficients because a large coefficient is a sign that a model is trying to stretch the coefficient to fit the training data. The problem is that the size of a coefficient will vary depending on the range of the feature it is representing.  
Features with small ranges have large coefficients. (0.1 * 10 = 1)  
Features with large ranges have small coefficients. (10 * 0.1 = 1)  
Ridge regression would therefore have a bias against features with small ranges, as they always have bigger coefficients. This is not helpful.  
To combat this we center all features around 0 by setting their mean to 0, and scale them having a standard deviation of 1. This makes all features comparable to each other.  

**Should we put StandardScaler() before or after PolynomialFeatures()? What is the difference and does it really matter? (Test it!)**


In [ ]:
# Testing


d = 10 # Testing for higher dimensions since that is where the differences in coefficients will be bigger
a = 0.1

model_a = Pipeline([
    ("polynomial", PolynomialFeatures(d)),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(a)),
])

model_b = Pipeline([
    ("scaler", StandardScaler()),
    ("polynomial", PolynomialFeatures(d)),
    ("ridge", Ridge(a)),
])

model_a.fit(X_train, y_train)
model_b.fit(X_train, y_train)

y_pred_a = model_a.predict(X_test)
y_pred_b = model_b.predict(X_test)

MSE_a = mean_squared_error(y_test, y_pred_a)
MSE_b = mean_squared_error(y_test, y_pred_b)

print(f"StandardScaler() after PolynomialFeatures() gives MSE = {MSE_a}")
print(f"StandardScaler() before PolynomialFeatures() gives MSE = {MSE_b}")

StandardScaler() after PolynomialFeatures() gives MSE = 1.0162410506976511
StandardScaler() before PolynomialFeatures() gives MSE = 1.3418512735445032


We can see from the result that putting StandardScaler() after PolynomialFeatures() gives a better result. The reason for this that changing the order changes what is actually being scaled.  
If you put the StandardScaler() before PolynomialFeatures() you standardize the raw features x1 and x2 in this case. These features then go into PolynomialFeatures() which gives you back 1, x1, x2, x1*x2, x1^2, x2^2. These do not have the same scales.  
If you put the StandardScaler() after PolynomialFeatures() you standardize all the combinations of the raw features. This means that 1, x1, x2, x1*x2, x1^2 and x2^2 will all have the same scales.  
This does not look dramatic at d=3 (which is why I upped my testing to d=10), but the higher the number of dimensions you have, the more dramatic the difference. This is because of how ridge regularization works, by penalizing large coefficients.